# Eclipse — HalfKAv2 NNUE Training (warm-restart fine-tune)

Fine-tunes the deployed wide net (`2048x2 -> 1024 -> 256 -> 1`, ep2c44) on the
quiet-filtered, endgame-weighted Stockfish depth-22 data.

**Inputs (all mounted, no secrets needed)**
* `simbae11/eclipse-chunks-quiet` — 194 HKAV2BIN chunk files (~3.9M positions each),
  `val_holdout.bin` (leak-free), `val_legacy.bin` (old-data forgetting monitor),
  `init_weights.pt` (ep2c44, dequantized byte-exact from the .nnue).
* `CKPT_DATASET` (cell 6) — added by `dev/scripts/train_babysit.sh` after the first
  session: carries `halfkav2.pt` + `resume_state.pt` between 12 h sessions.
* Hardware: T4 x2 (`machine_shape: NvidiaTeslaT4` in kernel-metadata.json).

**Per-chunk log line**: `train`, `val` (quiet holdout), `val_legacy`, val by men count
(`men[<=12 13-18 19-25 26+]`), lr, step, pos/s. A session ends with either
`All done!` (run complete) or `=== SESSION STOP (...)` (babysitter resumes it).

**Output** `/kaggle/working/halfkav2.pt` (latest), `resume_state.pt`, and
`halfkav2_epochE_chunkC.pt` every 24 chunks. Convert with
`dev/scripts/convert_halfkav2_nnue.py from-torch --output-cp-per-unit 300`.


In [ ]:
# Stream output live: Kaggle's committed (Save & Run All) runs otherwise
# batch stdout, and Python block-buffers when not a TTY. Line-buffer both
# streams so prints (chunk progress, k pos/s, val loss) appear immediately.
import sys
for _s in (sys.stdout, sys.stderr):
    try: _s.reconfigure(line_buffering=True)
    except Exception: pass

# Environment check
import sys, platform
import torch
print(f'python   {platform.python_version()}')
print(f'torch    {torch.__version__}')
print(f'cuda     {torch.cuda.is_available()} ({torch.version.cuda})')
if torch.cuda.is_available():
    print(f'gpu      {torch.cuda.get_device_name(0)}')
    print(f'vram     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Locate eval_training text data anywhere under /kaggle/input/, IF present.
# With premade HKAV2BIN chunks attached (eclipse-chunks-a) the text source is
# NOT required: training reads the chunks directly and the val set is sliced
# from premade chunk 0 (see next cell). DATA_PATH stays None when absent, and
# preprocess_chunk() (the live FEN->bin fallback) is simply never called.
import os
from pathlib import Path

matches = (sorted(Path('/kaggle/input').glob('**/eval_training.txt.gz')) or
           sorted(Path('/kaggle/input').glob('**/eval_training.txt')))
if matches:
    DATA_PATH = matches[0]
    print(f'training text source: {DATA_PATH} ({DATA_PATH.stat().st_size/1e6:.1f} MB)')
else:
    DATA_PATH = None
    print('no eval_training.txt(.gz) under /kaggle/input -- '
          'relying on premade chunks only (live preprocessing disabled)')


In [ ]:
# HalfKAv2 feature indexing -- exact mirror of src/nnue.cpp::feature_index
# and scripts/train_halfkav2.py. Drift here = silently wrong inference.
import gzip, math, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

FT_IN_FEATURES = 45056          # HalfKAv2: 64 (king sq) * 64 (piece sq) * 11 (piece slots)
# ── Architecture ──────────────────────────────────────────────────────────
# Stockfish-modern shape, trained from random init (see the header cell for why
# the 2048x2->1024->256 net is being retired rather than fine-tuned).
#
# DEPLOY NOTE: these four numbers are the contract with the C++ engine. Before a
# net trained here will load:
#     src/accumulator.hpp : kFtOutSize  = FT_OUT
#     src/nnue.hpp        : kL1OutSize  = L1_OUT
#                           kL2OutSize  = L2_OUT
# (kL1InSize follows kFtOutSize automatically; the SIMD kernels in src/nnue.cpp
# already handle these widths.) convert_halfkav2_nnue.py reads the widths out of
# the checkpoint, so it needs no edit. The loader cross-checks the header and
# refuses a mismatched file rather than evaluating garbage.
#
# ARCH picks a preset so switching lineages is one edit, not three:
#   'wide' -- 2048x2 -> 1024 -> 256: the DEPLOYED net (eclipse_v9_ep2c44) and the
#             shape the engine's headers are built for today. Used for the
#             2026-11 warm restart on the quiet-filtered data.
#   'sf16' -- 1024x2 -> 16 -> 32: the from-scratch experiment of Oct 2026
#             (checkpoint lineage simbae11/eclipse-checkpoint-sf16). Needs the
#             C++ header edits above before it can be deployed.
ARCH = 'wide'
FT_OUT, L1_OUT, L2_OUT = {'wide': (2048, 1024, 256),
                          'sf16': (1024, 16, 32)}[ARCH]
L1_IN          = 2 * FT_OUT     # concat of both perspectives
L3_OUT         = 1              # Value head: single logit (sigmoid = win prob)
N_PIECE_SLOTS  = 11

_PIECE_LETTER_TO_TYPE = {'p':1, 'n':2, 'b':3, 'r':4, 'q':5, 'k':6}
_FLIP_RANK = lambda s: s ^ 56

def parse_fen_board(fen_board):
    pieces = []
    sq = 56
    for ch in fen_board:
        if ch == '/':
            sq -= 16
            continue
        if ch.isdigit():
            sq += int(ch)
            continue
        is_white = ch.isupper()
        pt = _PIECE_LETTER_TO_TYPE[ch.lower()]
        pieces.append((sq, 0 if is_white else 1, pt))
        sq += 1
    return pieces

def feature_index(king_sq, piece_sq, pt, piece_is_ours):
    # ours: P=0,N=1,B=2,R=3,Q=4 (own king skipped). theirs: P=5..K=10.
    if piece_is_ours and pt == 6:
        raise ValueError('own king is not a HalfKAv2 feature')
    slot = (pt - 1) if piece_is_ours else (5 + (pt - 1))
    return king_sq * (64 * N_PIECE_SLOTS) + piece_sq * N_PIECE_SLOTS + slot

def fen_to_features(fen):
    parts = fen.split()
    board, stm = parts[0], parts[1]
    pieces = parse_fen_board(board)
    stm_color = 0 if stm == 'w' else 1
    king_sq = [None, None]
    for sq, color, pt in pieces:
        if pt == 6:
            king_sq[color] = sq
    if king_sq[0] is None or king_sq[1] is None:
        raise ValueError(f'FEN missing a king: {fen}')
    indices = [[], []]
    for persp in (0, 1):
        ksq = king_sq[persp] if persp == 0 else _FLIP_RANK(king_sq[persp])
        for sq, color, pt in pieces:
            is_ours = (color == persp)
            if is_ours and pt == 6:
                continue
            psq = sq if persp == 0 else _FLIP_RANK(sq)
            indices[persp].append(feature_index(ksq, psq, pt, is_ours))
    return (indices[0], indices[1]) if stm_color == 0 else (indices[1], indices[0])

In [ ]:
# Dataset helpers — _open_data handles .gz transparently; _parse_line is shared.
# HalfKAv2Dataset: in-memory, for the val set (≤ a few million positions).
# HalfKAv2StreamDataset: reservoir-shuffle streaming, for the full training set.
# HalfKAv2BinaryDataset: memmap random-access, used after preprocessing.
import struct as _struct
import torch
from torch.utils.data import DataLoader, Dataset, IterableDataset
MAX_ACTIVE = 32

def _open_data(path):
    if str(path).endswith('.gz'):
        return gzip.open(path, 'rt', encoding='utf-8')
    return open(path, encoding='utf-8')

def _parse_line(line, cp_scale):
    line = line.strip()
    if not line or line.startswith('#'):
        return None
    parts = line.split(';')
    if len(parts) == 4:
        try:
            fen      = parts[0].strip()
            win_prob = float(parts[1]) + 0.5 * float(parts[2])
        except ValueError:
            return None
    elif len(parts) == 2:
        try:
            fen      = parts[0].strip()
            win_prob = 1.0 / (1.0 + math.exp(-float(parts[1]) / cp_scale))
        except ValueError:
            return None
    else:
        return None
    try:
        us, them = fen_to_features(fen)
    except (KeyError, IndexError, ValueError):
        return None
    import numpy as _np
    us_arr   = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    them_arr = _np.full(MAX_ACTIVE, FT_IN_FEATURES, dtype=_np.int32)
    us_arr[:min(len(us),   MAX_ACTIVE)] = us[:MAX_ACTIVE]
    them_arr[:min(len(them), MAX_ACTIVE)] = them[:MAX_ACTIVE]
    return us_arr, them_arr, _np.float32(win_prob)

class HalfKAv2Dataset(Dataset):
    """In-memory dataset. Used for the val set (first val_size lines)."""
    def __init__(self, path, max_lines=None, cp_scale=410.0):
        import numpy as _np
        n = max_lines or self._count(path)
        if n == 0:
            raise RuntimeError(f'no rows in {path}')
        self.us_idx   = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.them_idx = _np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        self.targets  = _np.zeros((n, 1), dtype=_np.float32)
        i = 0
        with _open_data(path) as f:
            for line in f:
                if i >= n: break
                p = _parse_line(line, cp_scale)
                if p is None: continue
                self.us_idx[i], self.them_idx[i], self.targets[i, 0] = p
                i += 1
        if i < n:
            self.us_idx   = self.us_idx[:i]
            self.them_idx = self.them_idx[:i]
            self.targets  = self.targets[:i]
        print(f'val: loaded {i} positions')
    @staticmethod
    def _count(path):
        n = 0
        with _open_data(path) as f:
            for l in f:
                s = l.strip()
                if s and not s.startswith('#'): n += 1
        return n
    def __len__(self):  return len(self.targets)
    def __getitem__(self, i): return self.us_idx[i], self.them_idx[i], self.targets[i]

class HalfKAv2StreamDataset(IterableDataset):
    """Streaming dataset with reservoir shuffle buffer.
    Memory per worker: shuffle_buffer * (32+32)*4 + shuffle_buffer*4 bytes (~133 MB at 500k).
    max_lines is the TOTAL cap across all workers; each worker gets max_lines // n_workers.
    """
    def __init__(self, path, skip_lines=0, max_lines=None,
                 cp_scale=410.0, shuffle_buffer=500_000):
        self.path, self.skip_lines = path, skip_lines
        self.max_lines, self.cp_scale = max_lines, cp_scale
        self.shuffle_buffer = shuffle_buffer
    def __iter__(self):
        import numpy as _np, random as _rnd, torch as _torch
        wi = _torch.utils.data.get_worker_info()
        wid, nw = (wi.id, wi.num_workers) if wi else (0, 1)
        per_worker_max = ((self.max_lines + nw - 1) // nw
                          if self.max_lines is not None else None)
        buf_us   = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_them = _np.full((self.shuffle_buffer, MAX_ACTIVE), FT_IN_FEATURES, dtype=_np.int32)
        buf_tgt  = _np.zeros(self.shuffle_buffer, dtype=_np.float32)
        fill = 0; emitted = 0; didx = 0
        with _open_data(self.path) as f:
            if self.skip_lines:
                sk = 0
                for raw in f:
                    if raw.strip() and not raw.strip().startswith('#'):
                        sk += 1
                        if sk >= self.skip_lines: break
            for raw in f:
                if per_worker_max is not None and emitted >= per_worker_max: break
                stripped = raw.strip()
                if not stripped or stripped.startswith('#'): continue
                # Stride check BEFORE parsing: only parse lines this worker owns.
                if didx % nw != wid: didx += 1; continue
                didx += 1
                p = _parse_line(raw, self.cp_scale)
                if p is None: continue
                us_a, th_a, wp = p
                if fill < self.shuffle_buffer:
                    buf_us[fill]=us_a; buf_them[fill]=th_a; buf_tgt[fill]=wp; fill+=1
                else:
                    idx = _rnd.randrange(fill)
                    yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
                    buf_us[idx]=us_a; buf_them[idx]=th_a; buf_tgt[idx]=wp; emitted+=1
        perm = list(range(fill)); _rnd.shuffle(perm)
        for idx in perm:
            if per_worker_max is not None and emitted >= per_worker_max: break
            yield buf_us[idx].copy(), buf_them[idx].copy(), buf_tgt[idx:idx+1].copy()
            emitted += 1

# Binary format constants — must match preprocess_halfkav2.py
_BIN_MAGIC  = b'HKAV2BIN'
_BIN_HSIZE  = 16
_BIN_DTYPE  = np.dtype([('us',  np.uint16, (MAX_ACTIVE,)),
                         ('them',np.uint16, (MAX_ACTIVE,)),
                         ('tgt', np.float16)])

class HalfKAv2BinaryDataset(Dataset):
    """Memmap dataset from preprocess_halfkav2.py. Zero FEN parsing.
    Workers do array indexing only — use DataLoader(shuffle=True)."""
    def __init__(self, path, start=0, end=None):
        with open(path, 'rb') as f:
            if f.read(8) != _BIN_MAGIC: raise ValueError(f'{path}: not HKAV2BIN')
            n_total = _struct.unpack('<q', f.read(8))[0]
        self._mm  = np.memmap(path, dtype=_BIN_DTYPE, mode='r',
                              offset=_BIN_HSIZE, shape=(n_total,))
        self.start = start
        self.end   = n_total if end is None else min(end, n_total)
    def __len__(self): return self.end - self.start
    def __getitem__(self, i):
        r = self._mm[self.start + i]
        return (r['us'].astype(np.int64),
                r['them'].astype(np.int64),
                np.array([float(r['tgt'])], dtype=np.float32))

def collate(batch):
    import numpy as _np
    us   = _np.stack([b[0] for b in batch])
    them = _np.stack([b[1] for b in batch])
    tgt  = _np.stack([b[2] for b in batch])
    return (torch.from_numpy(us).long(),
            torch.from_numpy(them).long(),
            torch.from_numpy(tgt))

In [ ]:
# Model -- float mirror of the C++ inference path in src/nnue.cpp.
# Improvement 1: Quantization-Aware Training (QAT).
#
# Eclipse deploys int16 FT weights and int8 hidden weights. Training in float32
# then converting creates a systematic error: every weight is rounded to the
# nearest quantization step, and the model never saw that rounding during
# training so it can't compensate for it.
#
# QAT inserts fake-quantization nodes (round → dequantize) in the forward pass
# using the Straight-Through Estimator (STE) for gradients. The model sees
# the same discretization error at training time as at inference time, so it
# learns weights that are robust to rounding.
#
# Quantization constants (must match nnue.hpp / convert_halfkav2_nnue.py):
#   FT weights/biases  → int16: stored = round(real * 127),  scale = 1/127
#   Hidden weights     → int8:  stored = round(real * 64),   scale = 1/64
#   Activations (CReLU output) are in [0, 1] = [0, 127/127], discretised to
#   128 levels: scale = 1/127, same as FT.

FT_QUANT     = 127       # kFtQuant in nnue.hpp
WEIGHT_SCALE = 64        # kWeightScale in nnue.hpp

FT_SCALE  = 1.0 / FT_QUANT
FT_LO     = -32768.0 / FT_QUANT
FT_HI     =  32767.0 / FT_QUANT
W_SCALE   = 1.0 / WEIGHT_SCALE
W_LO      = -128.0 / WEIGHT_SCALE   # = -2.0
W_HI      =  127.0 / WEIGHT_SCALE   # ≈ +1.984
ACT_SCALE = 1.0 / FT_QUANT
ACT_LO    = 0.0
ACT_HI    = 1.0


class FakeQuantSTE(torch.autograd.Function):
    """Fake quantization: round to grid in forward, identity (STE) in backward."""
    @staticmethod
    def forward(ctx, x, scale, lo, hi):
        return torch.clamp(torch.round(x / scale) * scale, lo, hi)

    @staticmethod
    def backward(ctx, grad):
        return grad, None, None, None  # STE: pass gradient through unchanged


def fake_quant(x, scale, lo, hi):
    return FakeQuantSTE.apply(x, scale, lo, hi)


class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)


class HalfKAv2Net(nn.Module):
    def __init__(self, qat: bool = False):
        super().__init__()
        self.qat = qat

        # Last embedding row is the padding sentinel: zeroed at init,
        # pinned to zero after every step so padded slots contribute 0.
        self.ft = nn.Embedding(FT_IN_FEATURES + 1, FT_OUT,
                               padding_idx=FT_IN_FEATURES)
        nn.init.uniform_(self.ft.weight, -0.01, 0.01)
        with torch.no_grad():
            self.ft.weight[FT_IN_FEATURES].zero_()
        self.ft_bias = nn.Parameter(torch.zeros(FT_OUT))
        self.l1  = nn.Linear(L1_IN, L1_OUT)
        self.l2  = nn.Linear(L1_OUT, L2_OUT)
        self.l3  = nn.Linear(L2_OUT, L3_OUT)
        self.act = ClippedReLU()

    def forward(self, us_idx, them_idx):
        if self.qat:
            # Fake-quantize FT weights to int16 grid before lookup.
            ft_w_q = fake_quant(self.ft.weight, FT_SCALE, FT_LO, FT_HI)
            ft_b_q = fake_quant(self.ft_bias,   FT_SCALE, FT_LO, FT_HI)
            us   = F.embedding(us_idx,   ft_w_q).sum(dim=1) + ft_b_q
            them = F.embedding(them_idx, ft_w_q).sum(dim=1) + ft_b_q
            # Fake-quantize activations to 128 discrete levels ([0, 127/127]).
            us   = fake_quant(torch.clamp(us,   0.0, 1.0), ACT_SCALE, ACT_LO, ACT_HI)
            them = fake_quant(torch.clamp(them, 0.0, 1.0), ACT_SCALE, ACT_LO, ACT_HI)
        else:
            us   = self.ft(us_idx).sum(dim=1)   + self.ft_bias
            them = self.ft(them_idx).sum(dim=1) + self.ft_bias
            us   = self.act(us)
            them = self.act(them)

        x = torch.cat([us, them], dim=1)

        if self.qat:
            l1_w = fake_quant(self.l1.weight, W_SCALE, W_LO, W_HI)
            x = fake_quant(torch.clamp(F.linear(x, l1_w, self.l1.bias), 0.0, 1.0),
                           ACT_SCALE, ACT_LO, ACT_HI)
            l2_w = fake_quant(self.l2.weight, W_SCALE, W_LO, W_HI)
            x = fake_quant(torch.clamp(F.linear(x, l2_w, self.l2.bias), 0.0, 1.0),
                           ACT_SCALE, ACT_LO, ACT_HI)
            l3_w = fake_quant(self.l3.weight, W_SCALE, W_LO, W_HI)
            return F.linear(x, l3_w, self.l3.bias)
        else:
            x = self.act(self.l1(x))
            x = self.act(self.l2(x))
            return self.l3(x)

    def enable_qat(self):
        self.qat = True

    def disable_qat(self):
        self.qat = False

    def export_state_dict_for_quantization(self):
        sd = {}
        ft_w = self.ft.weight.detach()
        sd['ft.weight'] = ft_w[:FT_IN_FEATURES].T.contiguous()
        sd['ft.bias']   = self.ft_bias.detach()
        sd['l1.weight'] = self.l1.weight.detach()
        sd['l1.bias']   = self.l1.bias.detach()
        sd['l2.weight'] = self.l2.weight.detach()
        sd['l2.bias']   = self.l2.bias.detach()
        sd['l3.weight'] = self.l3.weight.detach()
        sd['l3.bias']   = self.l3.bias.detach()
        return sd


# The net2widernet helper that used to live here is gone along with the widen
# itself: it referenced FT_OUT_SRC / L1_OUT_SRC / L2_OUT_SRC, which no longer
# exist, and this run trains from random init. Recover it from git history if a
# function-preserving widen is ever wanted again.


In [ ]:
# ── Chunked training: warm-restart fine-tune of the deployed wide net ───────
# Each loop iteration trains on ONE premade chunk FILE (eclipse-chunks-quiet:
# 194 files x ~3.9M positions = ~238 optimizer steps each), validates, saves.
#
# Cross-session carry-over (Kaggle sessions die at 12 h, /kaggle/working starts
# empty): the KAGGLE_API_KEY secret is NOT readable from CLI-pushed kernels
# (UserSecretsClient -> "Connection error", every time), so in-notebook sync is
# normally OFF. Instead the run stops itself cleanly before the 12 h kill
# (SESSION_MAX_S), dev/scripts/train_babysit.sh copies halfkav2.pt +
# resume_state.pt from the kernel OUTPUT into CKPT_DATASET and mounts it, and the
# next session's ckpt_download() picks them up from /kaggle/input (mount
# fallback). RUN_ID is stored in resume_state.pt and must match, so a session can
# never silently resume a different run's weights.
#
# History that this cell encodes -- do not undo without re-reading:
#  * lr: a warm restart of an already-converged net needs a GENTLE peak. v39
#    (Aug 2026, -> ep2c44, +62 Elo) used 1.5e-4. v48 (Nov 2026) used the
#    from-scratch 1e-3 and wasted 9 h: quiet val never beat the starting net,
#    legacy val got 0.013 worse within 2 chunks of reaching 1e-3.
#  * eta_min = lr/10, not 0: a full anneal regressed the net before.
#  * Data is read into RAM per chunk and batched with vectorised indexing. The
#    old per-sample memmap DataLoader page-faulted against the network-mounted
#    /kaggle/input and ran at ~2 s/step.

import multiprocessing as _mp, struct as _struct2, gzip as _gz2, time as _time2
import shutil, numpy as _np2, os, subprocess as _sp2, json as _json2, tempfile as _tmp2, glob, re
import threading as _th, queue as _queue
import torch, torch.nn as _nn
import torch.nn.functional as F
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
RUN_ID       = 'quiet-v2'      # lineage guard; bump for a NEW run (and a new CKPT_DATASET)
CKPT_DATASET = 'simbae11/eclipse-checkpoint-quiet-v2'   # carry-over dataset for THIS run
EPOCHS       = 2
SESSION_MAX_S = 11 * 3600      # stop cleanly before Kaggle's 12 h kill (leaves time to save + write output)
# Mechanics smoke test (0 = off). >0: train only SMOKE_STEPS steps per chunk and
# stop the session after SMOKE_CHUNKS chunks, exactly as a session-limit stop.
SMOKE_STEPS  = 0
SMOKE_CHUNKS = 0
CHUNK_SIZE   = 100_000_000     # LIVE-preprocessing fallback only (no premade chunks attached)
SYNC_EVERY   = 24              # API sync cadence, only when the secret IS available
INIT_WEIGHTS_GLOB = '/kaggle/input/**/init_weights.pt'  # warm-start weights for a fresh lineage
WORK_DIR     = Path('/kaggle/working')
BIN_PATH     = WORK_DIR / 'training.bin'
VAL_BIN      = WORK_DIR / 'val_fixed.bin'
OUT_PT       = WORK_DIR / 'halfkav2.pt'
RESUME_PT    = WORK_DIR / 'resume_state.pt'
VAL_SIZE     = 200_000

TRAIN_CFG = dict(
    lr           = 5e-5,    # warm-restart peak: 2026-11-08 GPU sweep from ep2c44, 1000 steps each --
                            # 5e-5 beat 1.5e-4 / 4e-4 on every val bucket AND on unseen SF19 game
                            # positions (MAE 198->180cp); higher LR = more forgetting. floor = lr/10
    batch_size   = 16384,
    warmup_steps = 500,
    cp_scale     = 300.0,
    max_cp       = 4000,
    num_workers  = 4,       # live-preprocessing pool only
    weight_decay = 1e-4,
    grad_clip    = 1.0,
    qat          = True,
)
_RUN_T0 = _time2.time()

# ── Kaggle API auth (usually unavailable on CLI-pushed kernels; see above) ────
CKPT_OWNER = CKPT_DATASET.split('/')[0]
CKPT_SLUG  = CKPT_DATASET.split('/')[1]
KAGGLE_AUTH = False
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    _val = None
    for _name in ('KAGGLE_API_KEY', 'KAGGLE_API_TOKEN'):
        try:
            _val = _secrets.get_secret(_name)
        except Exception:
            continue
        if _val:
            print(f'using Kaggle secret {_name}')
            break
    if not _val:
        raise RuntimeError('neither KAGGLE_API_KEY nor KAGGLE_API_TOKEN is readable')
    os.environ.setdefault('KAGGLE_API_TOKEN', _val)
    os.environ.setdefault('KAGGLE_KEY', _val)
    os.environ.setdefault('KAGGLE_USERNAME', CKPT_OWNER)
    KAGGLE_AUTH = True
except Exception as e:
    print(f'no Kaggle API secret ({e}) -- in-notebook sync off; '
          f'carry-over via babysitter + mounted {CKPT_DATASET}')

def _kaggle(args):
    return _sp2.run(['kaggle'] + args, capture_output=True, text=True)

def ckpt_download():
    """Get halfkav2.pt + resume_state.pt for THIS lineage into WORK_DIR: Kaggle API
    if authed, else the MOUNTED CKPT_DATASET (only paths under its own slug, so
    another dataset's halfkav2.pt can never be picked up)."""
    if KAGGLE_AUTH:
        print(f'Checking {CKPT_DATASET} via API...')
        r = _kaggle(['datasets', 'download', '-d', CKPT_DATASET, '-p', str(WORK_DIR), '--unzip', '--force'])
        if r.returncode == 0:
            print(f'  downloaded: {[p.name for p in (OUT_PT, RESUME_PT) if p.exists()]}')
    if not OUT_PT.exists():
        own = lambda name: [p for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True)
                            if f'/{CKPT_SLUG}/' in p]
        w, rs = own('halfkav2.pt'), own('resume_state.pt')
        if w:
            shutil.copy2(w[0], OUT_PT)
            print(f'  mount fallback: {w[0]} -> {OUT_PT.name}')
            if rs:
                shutil.copy2(rs[0], RESUME_PT)
                print(f'  mount fallback: {rs[0]} -> {RESUME_PT.name}')
        else:
            mounted = [p for p in glob.glob('/kaggle/input/**/', recursive=True)
                       if p.rstrip('/').endswith('/' + CKPT_SLUG)]
            if mounted:
                # The babysitter only mounts this dataset AFTER a session has
                # saved into it, so mounted-but-empty means a broken carry-over.
                # Warm-starting from ep2c44 here would silently restart the run.
                raise RuntimeError(f'{CKPT_DATASET} is mounted at {mounted[0]} but holds no halfkav2.pt '
                                   f'-- broken carry-over; refusing to restart from init weights')
            print(f'  {CKPT_DATASET} not mounted -- this is the first session of {RUN_ID}')

def ckpt_upload(message):
    if not KAGGLE_AUTH:
        return
    with _tmp2.TemporaryDirectory() as d:
        d = Path(d)
        for f in (OUT_PT, RESUME_PT):
            if f.exists():
                shutil.copy2(f, d / f.name)
        meta = {'title': CKPT_SLUG, 'id': CKPT_DATASET, 'licenses': [{'name': 'other'}]}
        (d / 'dataset-metadata.json').write_text(_json2.dumps(meta))
        _t0 = _time2.time()
        r = _kaggle(['datasets', 'version', '-p', str(d), '-m', message, '--dir-mode', 'tar'])
        if r.returncode != 0 and ('404' in (r.stderr or '') or 'not found' in (r.stderr or '').lower()):
            r = _kaggle(['datasets', 'create', '-p', str(d)])
        print(f'  checkpoint sync {"OK" if r.returncode == 0 else "FAILED"} '
              f'({_time2.time()-_t0:.0f}s): {message if r.returncode == 0 else (r.stderr or r.stdout)[:300]}')

ckpt_download()

if not OUT_PT.exists():
    _init = sorted(glob.glob(INIT_WEIGHTS_GLOB, recursive=True))
    if not _init:
        raise FileNotFoundError('no checkpoint for this lineage and no init_weights.pt mounted -- '
                                'refusing to train a warm restart from random init')
    shutil.copy2(_init[0], OUT_PT)
    print(f'warm restart: {_init[0]} -> {OUT_PT.name} (fresh optimizer, warmup to lr={TRAIN_CFG["lr"]:.1e})')

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory // 2**30} GB')
else:
    n_gpus = 0
    print('device: cpu')

# ── Live preprocessing fallback (only when no premade chunks are attached) ────
_MAGIC2   = b'HKAV2BIN'
_RECSIZE2 = 32*2 + 32*2 + 2
_BATCH2   = 50_000

def _prep_batch(args):
    lines, cp_scale, max_cp = args
    import math, numpy as np
    FT=45056; MA=32; NS=11
    _P={'p':1,'n':2,'b':3,'r':4,'q':5,'k':6}
    def _bd(b):
        p,sq=[],56
        for ch in b:
            if ch=='/': sq-=16; continue
            if ch.isdigit(): sq+=int(ch); continue
            p.append((sq,0 if ch.isupper() else 1,_P[ch.lower()])); sq+=1
        return p
    def _fi(ks,ps,pt,o):
        if o and pt==6: raise ValueError
        return ks*(64*NS)+ps*NS+((pt-1) if o else (5+pt-1))
    def _ff(fen):
        b,s=fen.split()[0],fen.split()[1]; pcs=_bd(b); sc=0 if s=='w' else 1
        ks=[None,None]
        for sq,c,pt in pcs:
            if pt==6: ks[c]=sq
        if None in ks: raise ValueError
        ix=[[],[]]
        for pp in(0,1):
            kq=ks[pp] if pp==0 else ks[pp]^56
            for sq,c,pt in pcs:
                if c==pp and pt==6: continue
                ix[pp].append(_fi(kq,sq if pp==0 else sq^56,pt,c==pp))
        return (ix[0],ix[1]) if sc==0 else (ix[1],ix[0])
    def _pl(line):
        line=line.strip()
        if not line or line.startswith('#'): return None
        p=line.split(';')
        if len(p)==4:
            try: fen=p[0].strip(); wp=float(p[1])+0.5*float(p[2])
            except: return None
        elif len(p) in (2, 3):          # <fen>;<cp>[;<weight>] (weight ignored live)
            try:
                fen=p[0].strip(); cp=float(p[1])
                if abs(cp) > max_cp: return None
                wp=1.0/(1.0+math.exp(-cp/cp_scale))
            except: return None
        else: return None
        try: us,them=_ff(fen)
        except: return None
        ua=np.full(MA,FT,dtype=np.uint16); ta=np.full(MA,FT,dtype=np.uint16)
        ua[:min(len(us),MA)]=us[:MA]; ta[:min(len(them),MA)]=them[:MA]
        return ua,ta,np.float16(wp)
    dt=np.dtype([('us',np.uint16,(MA,)),('them',np.uint16,(MA,)),('tgt',np.float16)])
    recs=[r for line in lines for r in [_pl(line)] if r]
    if not recs: return b''
    buf=np.empty(len(recs),dtype=dt)
    for i,(u,t,w) in enumerate(recs): buf[i]['us']=u; buf[i]['them']=t; buf[i]['tgt']=w
    return buf.tobytes()

def _batches2(path, skip, limit):
    op=_gz2.open if str(path).endswith('.gz') else open
    batch=[]; seen=0; taken=0
    with op(path,'rt',encoding='utf-8') as f:
        for line in f:
            s=line.strip()
            if s and not s.startswith('#'):
                if seen < skip: seen+=1; continue
                batch.append(line)
                if len(batch)==_BATCH2:
                    yield (batch, TRAIN_CFG['cp_scale'], TRAIN_CFG['max_cp'])
                    taken+=len(batch); batch=[]
                    if limit and taken>=limit: return
    if batch: yield (batch, TRAIN_CFG['cp_scale'], TRAIN_CFG['max_cp'])

def preprocess_chunk(skip, limit, out_path=None):
    out_path = out_path or BIN_PATH
    avail_gb = shutil.disk_usage(WORK_DIR).free / 1e9
    need_gb = (limit * _RECSIZE2) / 1e9 + 1.5
    if avail_gb < need_gb:
        raise RuntimeError(f'Not enough disk: need {need_gb:.1f} GB, have {avail_gb:.1f} GB free.')
    print(f'\n--- Preprocessing skip={skip:,} limit={limit:,} ---')
    total=0; t0=_time2.time()
    with open(out_path,'wb') as fout:
        fout.write(_MAGIC2); fout.write(_struct2.pack('<q',0))
        with _mp.Pool(TRAIN_CFG['num_workers']) as pool:
            for data in pool.imap(_prep_batch, _batches2(DATA_PATH, skip, limit), chunksize=4):
                if not data: continue
                nr=len(data)//_RECSIZE2
                if limit and total+nr>limit: nr=limit-total; data=data[:nr*_RECSIZE2]
                fout.write(data); total+=nr
                if limit and total>=limit: break
    with open(out_path,'r+b') as f: f.seek(8); f.write(_struct2.pack('<q',total))
    print(f'  done: {total:,} in {_time2.time()-t0:.0f}s')
    return total

# ── Premade chunks ────────────────────────────────────────────────────────────
PREMADE_CHUNKS = {}
for _p in glob.glob('/kaggle/input/**/eval_chunk_*.bin', recursive=True):
    _m = re.search(r'eval_chunk_(\d+)\.bin$', _p)
    if _m:
        PREMADE_CHUNKS[int(_m.group(1))] = Path(_p)
if PREMADE_CHUNKS:
    _dirs = {p.parent for p in PREMADE_CHUNKS.values()}
    if len(_dirs) > 1:
        raise RuntimeError(f'premade chunks in {len(_dirs)} directories {sorted(map(str,_dirs))}; attach exactly one chunk dataset')
    if sorted(PREMADE_CHUNKS) != list(range(len(PREMADE_CHUNKS))):
        raise RuntimeError(f'premade chunk indices are not contiguous 0..{len(PREMADE_CHUNKS)-1}')
    print(f'found {len(PREMADE_CHUNKS)} premade chunk(s) in {next(iter(_dirs))}')

def read_records(path):
    """Whole HKAV2BIN file -> RAM (one sequential read; no random page faults)."""
    with open(path, 'rb') as f:
        if f.read(8) != _MAGIC2:
            raise ValueError(f'{path}: not HKAV2BIN')
        n = _struct2.unpack('<q', f.read(8))[0]
        buf = f.read(n * _RECSIZE2)
    if len(buf) != n * _RECSIZE2:
        raise ValueError(f'{path}: header says {n:,} records, file holds {len(buf)//_RECSIZE2:,}')
    return _np2.frombuffer(buf, dtype=_BIN_DTYPE)

def load_chunk(chunk_idx, skip, limit):
    """-> (records ndarray or None at end of data, source path)."""
    premade = PREMADE_CHUNKS.get(chunk_idx)
    if premade is not None:
        return read_records(premade), premade
    if PREMADE_CHUNKS or DATA_PATH is None:
        return None, None                       # premade set exhausted -> end of epoch
    if preprocess_chunk(skip, limit) == 0:
        return None, None
    return read_records(BIN_PATH), BIN_PATH

class _Prefetch:
    """Reads the NEXT chunk into RAM on a background thread while this one trains."""
    def __init__(self): self._t = None; self._key = None; self._res = None
    def start(self, chunk_idx, skip):
        if not PREMADE_CHUNKS:                  # live preprocessing: no prefetch
            return
        def work():
            try: self._res = load_chunk(chunk_idx, skip, CHUNK_SIZE)
            except Exception as e: self._res = e
        self._key = chunk_idx; self._res = None
        self._t = _th.Thread(target=work, daemon=True); self._t.start()
    def get(self, chunk_idx, skip):
        if self._t is not None and self._key == chunk_idx:
            self._t.join(); res = self._res; self._t = None
            if isinstance(res, Exception): raise res
            return res
        return load_chunk(chunk_idx, skip, CHUNK_SIZE)
_prefetch = _Prefetch()

def _to_dev(r):
    pin = device.type == 'cuda'
    def t(a):
        x = torch.from_numpy(a)
        return (x.pin_memory() if pin else x).to(device, non_blocking=True)
    return t(r['us'].astype(_np2.int64)), t(r['them'].astype(_np2.int64)), t(r['tgt'].astype(_np2.float32)[:, None])

def batch_stream(arr, rng, bs, max_batches=None):
    """Shuffled full batches from an in-RAM chunk; index prep runs on a thread so
    the GPU never waits on numpy."""
    perm = rng.permutation(len(arr)); nb = len(arr) // bs
    if max_batches: nb = min(nb, max_batches)
    q = _queue.Queue(maxsize=6)
    def work():
        for b in range(nb):
            idx = _np2.sort(perm[b*bs:(b+1)*bs])
            r = arr[idx]
            q.put((r['us'].astype(_np2.int64), r['them'].astype(_np2.int64),
                   r['tgt'].astype(_np2.float32)[:, None]))
        q.put(None)
    _th.Thread(target=work, daemon=True).start()
    pin = device.type == 'cuda'
    while (item := q.get()) is not None:
        yield tuple(((torch.from_numpy(a).pin_memory() if pin else torch.from_numpy(a))
                     .to(device, non_blocking=True)) for a in item)

# ── Model ─────────────────────────────────────────────────────────────────────
base_net = HalfKAv2Net(qat=TRAIN_CFG['qat']).to(device)
sd = torch.load(OUT_PT, map_location=device)
if tuple(sd['ft.weight'].shape) != (FT_OUT, FT_IN_FEATURES):
    raise RuntimeError(f'{OUT_PT} has ft.weight {tuple(sd["ft.weight"].shape)}, ARCH={ARCH!r} wants '
                       f'{(FT_OUT, FT_IN_FEATURES)} -- wrong checkpoint/ARCH; refusing to continue')
ft_w = sd['ft.weight'].T
base_net.load_state_dict({
    'ft.weight': torch.cat([ft_w, torch.zeros(1, ft_w.shape[1], device=device)], dim=0),
    'ft_bias':   sd['ft.bias'],
    **{k: sd[k] for k in ('l1.weight','l1.bias','l2.weight','l2.bias','l3.weight','l3.bias')},
})
del sd, ft_w
print(f'weights loaded from {OUT_PT.name}: {FT_IN_FEATURES} -> {FT_OUT}x2 -> {L1_OUT} -> {L2_OUT} -> {L3_OUT} '
      f'({sum(p.numel() for p in base_net.parameters())/1e6:.1f}M params)')

if n_gpus > 1:
    net = _nn.DataParallel(base_net)
    print(f'DataParallel: {n_gpus} GPUs, {TRAIN_CFG["batch_size"]//n_gpus} samples/GPU')
else:
    net = base_net

opt    = torch.optim.Adam(base_net.parameters(), lr=TRAIN_CFG['lr'], weight_decay=TRAIN_CFG['weight_decay'])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

# cosine.step() fires once per chunk FILE -> T_max counts chunk files.
_chunks_per_epoch = (len(PREMADE_CHUNKS) if PREMADE_CHUNKS
                     else (685_000_000 + CHUNK_SIZE - 1) // CHUNK_SIZE)
_est_chunks = EPOCHS * _chunks_per_epoch
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, _est_chunks), eta_min=TRAIN_CFG['lr'] / 10)
print(f'LR schedule: peak {TRAIN_CFG["lr"]:.1e} after {TRAIN_CFG["warmup_steps"]} warmup steps, cosine over '
      f'{_chunks_per_epoch} chunks/epoch x {EPOCHS} epochs = {_est_chunks} chunk steps, floor {TRAIN_CFG["lr"]/10:.1e}')

# ── Resume (epoch/chunk/optimizer/schedule) ───────────────────────────────────
global_step = 0; start_epoch = 0; start_chunk = 0
if RESUME_PT.exists():
    rs = torch.load(RESUME_PT, map_location='cpu')
    if rs.get('run_id') != RUN_ID:
        raise RuntimeError(f'resume_state.pt belongs to run {rs.get("run_id")!r}, this is {RUN_ID!r}. '
                           f'Mounted the wrong checkpoint dataset? Refusing to resume.')
    opt.load_state_dict(rs['optimizer'])
    cosine.load_state_dict(rs['cosine'])
    cosine.T_max   = max(1, _est_chunks)          # re-pin: load_state_dict restores the OLD shape
    cosine.eta_min = TRAIN_CFG['lr'] / 10
    scaler.load_state_dict(rs['scaler'])
    global_step = rs['global_step']
    start_epoch = rs['epoch']
    start_chunk = rs['chunk_idx'] + 1
    if PREMADE_CHUNKS and start_chunk >= _chunks_per_epoch:
        start_epoch += 1; start_chunk = 0
    print(f'RESUMING {RUN_ID}: epoch {start_epoch} chunk {start_chunk}  global_step {global_step:,}  '
          f'lr {opt.param_groups[0]["lr"]:.2e}  last val {rs.get("val")}')
    del rs
else:
    print(f'fresh optimizer for {RUN_ID} (no resume_state.pt)')

# ── Validation sets (in RAM) ──────────────────────────────────────────────────
_holdout = sorted(glob.glob('/kaggle/input/**/val_holdout.bin', recursive=True))
_legacy  = sorted(glob.glob('/kaggle/input/**/val_legacy.bin', recursive=True))
if _holdout:
    VAL_REC = read_records(_holdout[0]); print(f'val: leak-free holdout {_holdout[0]} ({len(VAL_REC):,})')
elif PREMADE_CHUNKS:
    VAL_REC = read_records(PREMADE_CHUNKS[0])[:VAL_SIZE]; print(f'val: first {len(VAL_REC):,} of chunk 0 (LEAKY)')
elif DATA_PATH is not None:
    preprocess_chunk(0, VAL_SIZE, VAL_BIN); VAL_REC = read_records(VAL_BIN)
else:
    raise FileNotFoundError('no val_holdout.bin, no premade chunk 0, no text source')
LEG_REC = read_records(_legacy[0]) if _legacy else None
if LEG_REC is not None:
    print(f'val_legacy (forgetting monitor, old unfiltered data): {_legacy[0]} ({len(LEG_REC):,})')
_men = (VAL_REC['us'] != FT_IN_FEATURES).sum(1) + 1   # features exclude own king
VAL_BUCKETS = {'<=12': _men <= 12, '13-18': (_men > 12) & (_men <= 18),
               '19-25': (_men > 18) & (_men <= 25), '26+': _men > 25}

@torch.no_grad()
def _per_pos_loss(arr):
    net.eval(); out = []
    for i in range(0, len(arr), TRAIN_CFG['batch_size']):
        us, them, t = _to_dev(arr[i:i + TRAIN_CFG['batch_size']])
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
            lg = net(us, them)
        out.append(F.binary_cross_entropy_with_logits(lg.float(), t, reduction='none').squeeze(1).cpu())
    net.train()
    return torch.cat(out).numpy()

def evaluate():
    lv = _per_pos_loss(VAL_REC)
    r = {'val': float(lv.mean())}
    if LEG_REC is not None:
        r['val_legacy'] = float(_per_pos_loss(LEG_REC).mean())
    for k, m in VAL_BUCKETS.items():
        r['v' + k] = float(lv[m].mean())
    return r

def fmt_eval(r):
    return (f'val={r["val"]:.5f}  ' + (f'val_legacy={r["val_legacy"]:.5f}  ' if 'val_legacy' in r else '')
            + 'men[' + ' '.join(f'{k}:{r["v"+k]:.5f}' for k in VAL_BUCKETS) + ']')

_session_start_eval = evaluate()
print(f'\nsession-start eval (= previous chunk if resuming, ep2c44 if fresh): {fmt_eval(_session_start_eval)}')

# ── Training loop ─────────────────────────────────────────────────────────────
_pending_sync = False
_stop_reason  = None
_chunk_times  = []
_smoke_chunks = 0
if start_epoch >= EPOCHS:
    print('\nTraining already complete (resume_state past EPOCHS).')

for epoch in range(start_epoch, EPOCHS):
    print(f'\n\n#################### EPOCH {epoch} ####################')
    chunk_idx = start_chunk if epoch == start_epoch else 0
    while True:
        t_chunk = _time2.time()
        arr, src = _prefetch.get(chunk_idx, chunk_idx * CHUNK_SIZE)
        if arr is None:
            print(f'  end of data at chunk {chunk_idx} -- epoch {epoch} complete.')
            if _pending_sync:
                ckpt_upload(f'{RUN_ID} epoch{epoch} end step{global_step:,}'); _pending_sync = False
            break
        _prefetch.start(chunk_idx + 1, (chunk_idx + 1) * CHUNK_SIZE)
        print(f'\n=== Epoch {epoch}  Chunk {chunk_idx}/{_chunks_per_epoch - 1}  ({len(arr):,} positions, {Path(src).name}) ===')

        # Deterministic per-(run, epoch, chunk) shuffle: a resumed session replays nothing.
        rng = _np2.random.default_rng([sum(map(ord, RUN_ID)), epoch, chunk_idx])
        net.train()
        ep_loss = 0.0; nbatch = 0; t_start = _time2.time()
        for us, them, target in batch_stream(arr, rng, TRAIN_CFG['batch_size'],
                                             max_batches=SMOKE_STEPS or None):
            if global_step < TRAIN_CFG['warmup_steps']:
                for g in opt.param_groups:
                    g['lr'] = TRAIN_CFG['lr'] * (global_step + 1) / TRAIN_CFG['warmup_steps']
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
                logits = net(us, them)
                loss   = F.binary_cross_entropy_with_logits(logits.squeeze(1), target.squeeze(1))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(base_net.parameters(), TRAIN_CFG['grad_clip'])
            scaler.step(opt)
            scaler.update()
            with torch.no_grad():
                base_net.ft.weight[FT_IN_FEATURES].zero_()
            ep_loss += loss.item(); nbatch += 1; global_step += 1
        if nbatch == 0 or not _np2.isfinite(ep_loss):
            raise RuntimeError(f'TRAINING ABORTED: chunk {chunk_idx} gave nbatch={nbatch} loss={ep_loss}')
        t_train = _time2.time() - t_start

        ev = evaluate()
        cosine.step()
        _chunk_times.append(_time2.time() - t_chunk)
        print(f'\n  Epoch {epoch} Chunk {chunk_idx}: train={ep_loss/nbatch:.5f}  {fmt_eval(ev)}  '
              f'lr={opt.param_groups[0]["lr"]:.2e}  step={global_step:,}  '
              f'{nbatch*TRAIN_CFG["batch_size"]/t_train/1e3:.0f}k pos/s  time={_chunk_times[-1]/60:.1f}min')

        # ── Save weights + training position ──
        _sd = base_net.export_state_dict_for_quantization()
        torch.save(_sd, OUT_PT)
        if (chunk_idx + 1) % 24 == 0:          # keep a sparse trail of named checkpoints for later SPRTs
            torch.save(_sd, WORK_DIR / f'halfkav2_epoch{epoch}_chunk{chunk_idx}.pt')
        del _sd
        torch.save(dict(run_id=RUN_ID, epoch=epoch, chunk_idx=chunk_idx, global_step=global_step,
                        optimizer=opt.state_dict(), cosine=cosine.state_dict(),
                        scaler=scaler.state_dict(), val=ev), RESUME_PT)
        _pending_sync = True
        if KAGGLE_AUTH and (chunk_idx == 0 or (chunk_idx + 1) % SYNC_EVERY == 0):
            ckpt_upload(f'{RUN_ID} epoch{epoch} chunk{chunk_idx} step{global_step:,} val={ev["val"]:.5f}')
            _pending_sync = False
        if src == BIN_PATH and BIN_PATH.exists():
            BIN_PATH.unlink()
        del arr

        # ── Session budget: stop cleanly with room for one more chunk + output ──
        elapsed = _time2.time() - _RUN_T0
        if SMOKE_STEPS:
            _smoke_chunks += 1
            if _smoke_chunks >= max(1, SMOKE_CHUNKS):
                _stop_reason = f'smoke test: {_smoke_chunks} chunk(s) x {SMOKE_STEPS} steps'
        elif elapsed + 1.5 * max(_chunk_times[-3:]) + 300 > SESSION_MAX_S:
            _stop_reason = f'session budget ({elapsed/3600:.2f} h used of {SESSION_MAX_S/3600:.1f} h)'
        if _stop_reason:
            break
        chunk_idx += 1
    if _stop_reason:
        break

if _pending_sync:
    ckpt_upload(f'{RUN_ID} session end step{global_step:,}')
if _stop_reason:
    # NOT "All done!" -- the babysitter treats that string as run completion.
    print(f'\n=== SESSION STOP ({_stop_reason}) at epoch {epoch} chunk {chunk_idx}, step {global_step:,}; '
          f'resume needed: babysitter carries halfkav2.pt + resume_state.pt into {CKPT_DATASET} ===')
else:
    print('\nAll done!')


## Saving the artifact off Kaggle

1. Click **Save Version → Save & Run All**. The notebook output (including `/kaggle/working/halfkav2.pt`) is captured on the version that finishes successfully.
2. Open the saved version → **Output** tab → click `halfkav2.pt` → Download.
3. Locally, drop it into `data/` and run the convert command above. See `KAGGLE.md` for verification steps and saturation-warning interpretation.